In [24]:
# ── Configuration ─────────────────────────────────────────────────────
# Edit these paths to point to your actual CSV files

basepath = Path("/Users/loukia/UCL Dropbox/Loukia Katsouri/DataProtocolsEquipment/Protein work/NULISA/hTau-12mo_08092025")
FILE1 = basepath / "20260115_hTaugenotypes_NULISA_Aug2025.csv"          # path to first CSV  (unique MouseID per row)
FILE2 = basepath / "20260115_hTauNLGF_sample_annotation.csv"          # path to second CSV  (may have duplicate MouseID, e.g. multiple tissues)
OUT   = basepath / "20260115_hTauNLGF_sample_annotation_merged.csv"         # output CSV path
HOW   = "outer"               # join type: "left", "inner", "right", or "outer" )I chose outer:Everything from both files	Unmatched rows from either side are kept, with NaN for the missing columns

# ── Imports ───────────────────────────────────────────────────────────
from pathlib import Path
import pandas as pd


In [25]:
def read_csv_safe(path) -> pd.DataFrame:
    """Read a CSV/TSV, auto-detecting encoding and delimiter."""
    from pathlib import Path
    path = Path(path)

    for enc in ("utf-8-sig", "utf-16", "latin-1"):
        for sep in (None, "\t", ","):          # None = Python engine sniffs
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python" if sep is None else "c")
                # Sanity check: if sniffing worked we should have >1 column
                if len(df.columns) > 1:
                    return df
            except (UnicodeDecodeError, UnicodeError):
                break                          # wrong encoding, try next enc
            except (pd.errors.EmptyDataError, pd.errors.ParserError):
                continue
            except FileNotFoundError:
                raise FileNotFoundError(f"File not found: {path}")

    # Last-resort attempt
    raise ValueError(f"Could not parse '{path}' — check encoding and delimiter")


def require_mouseid(df: pd.DataFrame, label: str) -> None:
    """Validate that the DataFrame contains a 'MouseID' column."""
    if "MouseID" not in df.columns:
        cols_preview = ", ".join(map(str, list(df.columns)[:30]))
        raise KeyError(
            f"Missing required column 'MouseID' in {label}. "
            f"Columns found (first 30): {cols_preview}"
        )


def tissue_report(df2: pd.DataFrame) -> None:
    """If a 'tissue' column exists in file2, print duplicate-MouseID statistics."""
    # case-insensitive column search
    tissue_col = next((c for c in df2.columns if str(c).lower() == "tissue"), None)
    if tissue_col is None:
        return

    counts = df2["MouseID"].value_counts(dropna=False)
    multi = counts[counts > 1]

    print("\n[file2] Duplicate MouseID report:")
    print(f"  MouseIDs with >1 row: {len(multi):,}")

    if len(multi) == 0:
        return

    print("  Top 10 MouseIDs by count (with distinct tissue values):")
    for mouse_id, n in multi.head(10).items():
        tissues = (
            df2.loc[df2["MouseID"] == mouse_id, tissue_col]
            .dropna().astype(str).unique().tolist()
        )
        tissues_str = ", ".join(tissues[:12])
        if len(tissues) > 12:
            tissues_str += ", ..."
        print(f"    {mouse_id}: {n} | tissues: {tissues_str}")

In [29]:
# ── Read CSVs ─────────────────────────────────────────────────────────
df1 = read_csv_safe(FILE1)
df2 = read_csv_safe(FILE2)

# Validate that both have MouseID
require_mouseid(df1, "file1")
require_mouseid(df2, "file2")

# Harmonise MouseID dtype (one file has int, the other text)
df1["MouseID"] = df1["MouseID"].astype(str).str.strip()
df2["MouseID"] = df2["MouseID"].astype(str).str.strip()

# ── Summary ───────────────────────────────────────────────────────────
print("[summary]")
print(f"  file1 rows: {len(df1):,} | unique MouseID: {df1['MouseID'].nunique(dropna=False):,}")
print(f"  file2 rows: {len(df2):,} | unique MouseID: {df2['MouseID'].nunique(dropna=False):,}")
print(f"  join how: {HOW}")

# Optional tissue report (only prints if file2 has a 'tissue' column)
tissue_report(df2)

# ── Merge (one-to-many: file1 unique, file2 may have duplicates) ─────
merged = pd.merge(
    df2,
    df1,
    on="MouseID",
    how=HOW,
    # suffixes=("_file1", "_file2"),
    # validate="one_to_many",   # explicitly allows duplicates in file2
    suffixes=("_file2", "_file1"),   # ← swapped suffixes to match new order
    validate="many_to_one",          # ← flipped: left (df2) may duplicate, right (df1) unique
)


# ── Save ──────────────────────────────────────────────────────────────
out_path = Path(OUT)
merged.to_csv(out_path, index=False)

print(f"\n[output]")
print(f"  output rows: {len(merged):,}")
print(f"  wrote: {out_path.resolve()}")

# Preview the first few rows
merged.head()

[summary]
  file1 rows: 42 | unique MouseID: 42
  file2 rows: 86 | unique MouseID: 43
  join how: outer

[file2] Duplicate MouseID report:
  MouseIDs with >1 row: 43
  Top 10 MouseIDs by count (with distinct tissue values):
    control: 2 | tissues: control
    1119511: 2 | tissues: HPC, CTX
    1118777: 2 | tissues: HPC, CTX
    1120950: 2 | tissues: HPC, CTX
    1119156: 2 | tissues: HPC, CTX
    1121069: 2 | tissues: HPC, CTX
    1120880: 2 | tissues: CTX, HPC
    1119959: 2 | tissues: HPC, CTX
    1120159: 2 | tissues: CTX, HPC
    1119723: 2 | tissues: HPC, CTX

[output]
  output rows: 86
  wrote: /Users/loukia/UCL Dropbox/Loukia Katsouri/DataProtocolsEquipment/Protein work/NULISA/hTau-12mo_08092025/20260115_hTauNLGF_sample_annotation_merged.csv


,plateID,sampleName,AUTO_WELLPOSITION,Well,SAMPLE_MATRIX,MouseID,Tissue,Genotype_file2,Sex_file2,ID,Mutation 1,Genotype 1,Mutation 2,Genotype 2,Mutation 3,Genotype 3,Sex_file1,Genotype_file1
0,20250910_BF362 Loukia Katsouri NULISA Mouse CN...,1118777_HPC,C12,c12,OTHER,1118777,HPC,NaN,NaN,EAA-1118777,APPtm3,hom,MAPT,wt,muMapt,ko/ko,m,NLGF/MAPT KO
1,20250910_BF362 Loukia Katsouri NULISA Mouse CN...,1118777_CTX,D5,d5,OTHER,1118777,CTX,NaN,NaN,EAA-1118777,APPtm3,hom,MAPT,wt,muMapt,ko/ko,m,NLGF/MAPT KO
2,20250910_BF362 Loukia Katsouri NULISA Mouse CN...,1118783_CTX,A7,a7,OTHER,1118783,CTX,NaN,NaN,EAA-1118783,APPtm3,hom,MAPT,T,muMapt,ko/ko,f,NLGF/hTau
3,20250910_BF362 Loukia Katsouri NULISA Mouse CN...,1118783_HPC,E3,e3,OTHER,1118783,HPC,NaN,NaN,EAA-1118783,APPtm3,hom,MAPT,T,muMapt,ko/ko,f,NLGF/hTau
4,20250910_BF362 Loukia Katsouri NULISA Mouse CN...,1118785_CTX,A4,a4,OTHER,1118785,CTX,NaN,NaN,EAA-1118785,APPtm3,hom,MAPT,T,muMapt,ko/ko,f,NLGF/hTau
